# Práctica 7 - Train, Validation y Test
**Sistemas de Aprendizaje Automático · Sesión 7**   
**Archivo de datos:** `clientes_ecommerce.csv`

### Contexto
La empresa de comercio electrónico quiere predecir si un cliente comprará durante el próximo mes. En esta práctica separarás correctamente los datos, evitarás fugas de información y entrenarás un primer clasificador **sin utilizar el conjunto de test para evaluarlo**.

**Instrucciones:** completa las celdas con `TODO`, ejecuta el notebook en orden y contesta en Markdown las preguntas. No cambies la semilla `random_state=42`. El archivo CSV debe estar en la misma carpeta que el notebook.

**Objetos finales:** `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`, `preprocesador`, `modelo`, `predicciones_train`, `predicciones_val`, `accuracy_train` y `accuracy_val`.

## Parte 1 - Preparación del dataset
### Ejercicio 1. Cargar y explorar
Carga los datos, muestra las primeras cinco filas, las dimensiones y los tipos de datos. Comprueba que el CSV se encuentra junto al notebook.

In [ ]:
import pandas as pd
import numpy as np

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv('clientes_ecommerce.csv')

# TODO: primeras cinco filas, shape y dtypes
print(df.head())
print(df.shape)
print(df.dtypes)

**Responde:** ¿Cuántas muestras y columnas hay? ¿Qué variable queremos predecir? ¿Es clasificación o regresión?

**Respuesta:** 400 muestras y 12 columnas. Queremos predecir `compra_proximo_mes` y es un problema de clasificación binaria

### Ejercicio 2. Separar características y objetivo
Crea `X` excluyendo `cliente_id` y `compra_proximo_mes`. Crea `y` codificando `No` → 0 y `Sí` → 1. Comprueba dimensiones, valores ausentes en `y` y distribución de clases.

In [ ]:
# TODO: crear X sin identificador ni variable objetivo
X = df.drop(columns=['cliente_id', 'compra_proximo_mes'])

# TODO: crear y con map({"No": 0, "Sí": 1})
y = df['compra_proximo_mes'].map({"No": 0, "Sí": 1})

# TODO: comprobar X.shape, y.shape, y.isna().sum() y y.value_counts()
print(X.shape)
print(y.shape)
print(y.isna().sum())
print(y.value_counts())

**Responde:** ¿Cuántas características hay? ¿Por qué debemos excluir el identificador y el target de `X`?

**Respuesta:** Hay 10 características. Excluimos `cliente_id` porque es un código único sin poder predictivo que favorece el sobreajuste, y excluimos `compra_proximo_mes` para evitar fuga de datos (*data leakage*), ya que el modelo no puede ver la respuesta durante el entrenamiento

## Parte 2 - División de datos
### Ejercicio 3. Reservar test
Reserva el 20 % para test mediante `train_test_split` con `random_state=42` y `stratify=y`. El 80 % restante se llamará `X_temp`, `y_temp`.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: primera división
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# TODO: mostrar tamaños de los dos conjuntos
print("Temp:", X_temp.shape, y_temp.shape)
print("Test:", X_test.shape, y_test.shape)

**Responde:** ¿Para qué sirven `random_state` y `stratify`?

**Respuesta:** `random_state` fija la semilla aleatoria para que la partición sea reproducible y siempre salgan los mismos registros, mientras que `stratify` conserva la misma proporción de clases de `y` en ambos subconjuntos

### Ejercicio 4. Obtener validación
Divide `X_temp`, `y_temp` para que el reparto final sea **60 % train / 20 % validation / 20 % test**. Utiliza `random_state=42` y estratificación.

In [ ]:
# TODO: segunda división
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

**Responde:** ¿Qué valor de `test_size` necesitas en la segunda división? Demuéstralo con una operación.

**Respuesta:** Necesito `test_size=0.25`, porque `X_temp` representa el 80% del total (0.80) y queremos sacar un 20% del total (0.20) para validación: $0.20 / 0.80 = 0.25$

### Ejercicio 5. Comprobar dimensiones
Muestra `shape` de los seis conjuntos. Verifica que la suma de muestras coincide con el tamaño original. Completa: Train 240 muestras; Validation 80; Test 80.

In [ ]:
# TODO: imprimir shape de X_train, X_val, X_test, y_train, y_val, y_test
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

# TODO: comprobar la suma de tamaños
print("Suma total:", len(X_train) + len(X_val) + len(X_test))

## Parte 3 - Verificar los conjuntos
### Ejercicio 6. Distribución de clases
Calcula el porcentaje de las clases 0 y 1 en train, validation y test con `value_counts(normalize=True)`. Crea una tabla comparativa.

In [ ]:
# TODO: proporciones de clases de y_train, y_val y y_test
prop_train = y_train.value_counts(normalize=True) * 100
prop_val = y_val.value_counts(normalize=True) * 100
prop_test = y_test.value_counts(normalize=True) * 100

# TODO: tabla con columnas Train, Validation, Test
tabla_proporciones = pd.DataFrame({
    "Train": prop_train.round(2),
    "Validation": prop_val.round(2),
    "Test": prop_test.round(2)
})
tabla_proporciones

**Responde:** ¿Se conservan aproximadamente las proporciones? ¿Por qué importa cuando una clase es minoritaria?

**Respuesta:** Sí, se mantienen casi idénticas (alrededor de 61.5% para 0 y 38.5% para 1). Importa mucho porque si la clase minoritaria es pequeña, un reparto aleatorio sin estratificar podría dejar muy pocos ejemplos en validación o test, falseando tanto el aprendizaje como la evaluación del modelo

### Ejercicio 7. Independencia de conjuntos
Comprueba con intersecciones de índices que ninguna fila original está presente en dos conjuntos. Usa `assert`. Recuerda que esto **no** detecta automáticamente duplicados con índices distintos.

In [ ]:
# TODO: intersecciones de índices train/val, train/test y val/test
inter_train_val = X_train.index.intersection(X_val.index)
inter_train_test = X_train.index.intersection(X_test.index)
inter_val_test = X_val.index.intersection(X_test.index)

# TODO: tres assert para comprobar que están vacías
assert len(inter_train_val) == 0
assert len(inter_train_test) == 0
assert len(inter_val_test) == 0

**Responde:** ¿Qué problema habría si se compartieran muestras? ¿Qué limitación tiene esta comprobación?

**Respuesta:** Si se compartieran muestras el modelo evaluaría sobre datos que ya vio al entrenar, provocando sobreajuste y métricas artificialmente altas. Su limitación es que solo mira los índices de las filas, por lo que no detectaría registros duplicados con distinto índice

## Parte 4 - Data leakage y preprocesamiento
### Ejercicio 8. Detectar errores
Analiza estos casos **sin ejecutarlos**:

**A:** `scaler.fit_transform(X)` antes de dividir en train/test.

**B:** `imputer.fit_transform(X_train)` y `imputer.transform(X_val)`.

**C:** evaluar repetidamente en `X_test` y elegir los hiperparámetros con esa puntuación.

Para cada caso explica si es correcto, cuál es el riesgo y cómo lo corregirías.

**Caso A:** Incorrecto. Produce fuga de datos (*data leakage*) porque la media y desviación estándar se calculan usando también información de test. Lo corregiría dividiendo primero y ajustando el escalador solo con `X_train` (`fit_transform(X_train)` y `transform(X_test)`)

**Caso B:** Correcto. Se ajusta el imputador aprendiendo los estadísticos únicamente del conjunto de entrenamiento y se aplican esos mismos valores sobre validación sin reajustar

**Caso C:** Incorrecto. Si usamos test para ajustar hiperparámetros acabamos sobreajustando las decisiones al conjunto de prueba y pierde su validez como evaluación final. Lo corregiría usando el conjunto de validación para comparar configuraciones y dejando `X_test` intocable hasta el final

### Ejercicio 9. Edad anómala
Para este ejercicio, edades superiores a 120 se consideran inválidas. Sustitúyelas por `NaN` en **copias** de train, validation y test. Muestra el número de ausentes en `edad` de cada conjunto. No calcules la mediana: lo hará el pipeline.

In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

# TODO: aplicar regla de edad > 120 en los tres conjuntos
X_train.loc[X_train['edad'] > 120, 'edad'] = np.nan
X_val.loc[X_val['edad'] > 120, 'edad'] = np.nan
X_test.loc[X_test['edad'] > 120, 'edad'] = np.nan

# TODO: mostrar NaN en edad para train/val/test
print("Ausentes edad train:", X_train['edad'].isna().sum())
print("Ausentes edad val:", X_val['edad'].isna().sum())
print("Ausentes edad test:", X_test['edad'].isna().sum())

### Ejercicio 10. Preprocesamiento
Construye tres pipelines y combínalos en `preprocesador`:

- **Numéricas:** imputación mediana + `StandardScaler`.
- **Nominales:** imputación moda + `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.
- **Ordinal:** imputación moda + `OrdinalEncoder`, orden Bronce, Plata, Oro, Platino y desconocidos `-1`.

**No ajustes el preprocesador por separado con validation ni test.**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales", "gasto_mensual",
    "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# TODO: pipeline_numerico = Pipeline([...])
pipeline_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# TODO: pipeline_nominal = Pipeline([...])
pipeline_nominal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# TODO: pipeline_ordinal = Pipeline([...])
pipeline_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# TODO: preprocesador = ColumnTransformer([...])
preprocesador = ColumnTransformer([
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
])

**Responde:** ¿Qué ventajas tiene mantener todas las transformaciones en un único preprocesador?

**Respuesta:** Evita el *data leakage* al encapsular todo el ajuste en un solo paso sobre train, garantiza que se apliquen exactamente las mismas transformaciones y en el mismo orden a validation y test, y simplifica mucho el código para llevar el modelo a producción

## Parte 5 - Primer modelo
### Ejercicio 11. Pipeline completo
Construye `modelo` con dos etapas: `preprocesamiento` (el objeto anterior) y `clasificador` (`LogisticRegression(max_iter=1000)`).

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: modelo = Pipeline([...])
modelo = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(max_iter=1000))
])

**Responde:** ¿Qué ocurre antes de entrenar el clasificador? ¿Cómo se tratará una categoría nominal nueva?

**Respuesta:** Antes de entrenar la regresión logística se ejecuta automáticamente `preprocesamiento`, imputando los nulos, escalando las variables numéricas y codificando las categóricas. Si aparece una categoría nominal nueva que no estaba en entrenamiento, el `OneHotEncoder` con `handle_unknown="ignore"` la ignora sin lanzar error y rellena todas sus columnas binarias con 0

### Ejercicio 12. Entrenar
Entrena el pipeline **solo** con `X_train` e `y_train`.

In [ ]:
# TODO: modelo.fit(...)
modelo.fit(X_train, y_train)

**Responde:** ¿Qué aprende el preprocesador? ¿Qué aprende el clasificador?

**Respuesta:** El preprocesador aprende las medianas, modas, medias, desviaciones estándar y categorías únicas de `X_train`. El clasificador aprende los pesos (coeficientes) y el sesgo (*intercept*) de la regresión logística que mejor separan las clases 0 y 1

### Ejercicio 13. Predecir
Crea `predicciones_train` y `predicciones_val` usando `modelo.predict()`. Muestra las primeras diez predicciones de validación. **No uses `X_test`**.

In [ ]:
# TODO: predicciones_train = ...
predicciones_train = modelo.predict(X_train)

# TODO: predicciones_val = ...
predicciones_val = modelo.predict(X_val)

# TODO: primeras diez predicciones de validación
print(predicciones_val[:10])

**Responde:** ¿Qué significan 0 y 1? ¿Por qué no hay que transformar manualmente `X_val` antes de `predict()`?

**Respuesta:** `0` significa que el modelo predice que el cliente no comprará el próximo mes y `1` que sí comprará. No hace falta transformar `X_val` a mano porque `modelo` es un `Pipeline` que aplica automáticamente el paso `preprocesamiento` con `transform()` antes de predecir

### Ejercicio 14. Accuracy
Calcula `accuracy_train` y `accuracy_val`. Muestra una tabla con ambos resultados. No se permite calcular accuracy sobre test.

In [ ]:
from sklearn.metrics import accuracy_score

# TODO: accuracy_train = ...
accuracy_train = accuracy_score(y_train, predicciones_train)

# TODO: accuracy_val = ...
accuracy_val = accuracy_score(y_val, predicciones_val)

# TODO: tabla comparativa
tabla_accuracy = pd.DataFrame({
    "Conjunto": ["Train", "Validation"],
    "Accuracy": [round(accuracy_train, 4), round(accuracy_val, 4)]
})
tabla_accuracy

**Responde:** ¿Qué conjunto obtiene mayor accuracy? ¿Hay diferencia importante? ¿Una accuracy alta garantiza que el modelo sea bueno?

**Respuesta:** Obtiene mayor accuracy el conjunto de Train (0.7083 frente a 0.6625 en Validation). Hay una ligera diferencia de unos 4.5 puntos porcentuales, pero no es un sobreajuste grave. Una accuracy alta por sí sola no garantiza que el modelo sea bueno, especialmente en datasets desbalanceados donde acertar siempre la clase mayoritaria ya daría un porcentaje elevado sin detectar bien la clase minoritaria

### Ejercicio 15. Comparar predicciones
Crea un DataFrame `resultados` con las columnas `Real`, `Predicción` y `Correcta`. Muestra las primeras 15 filas y cuenta aciertos y errores.

In [ ]:
# TODO: resultados = pd.DataFrame({...})
resultados = pd.DataFrame({
    "Real": y_val,
    "Predicción": predicciones_val,
    "Correcta": y_val == predicciones_val
})

# TODO: primeras 15 filas
print(resultados.head(15))

# TODO: total de predicciones correctas e incorrectas
print("Aciertos:", resultados["Correcta"].sum())
print("Errores:", (~resultados["Correcta"]).sum())

**Responde:** ¿Qué información adicional necesitaríamos para analizar mejor los errores del clasificador?

**Respuesta:** Necesitaríamos una matriz de confusión para distinguir entre falsos positivos y falsos negativos, métricas específicas por clase como precisión, *recall* y F1-score, y las probabilidades predichas (`predict_proba`) junto con las características de las muestras falladas para ver en qué perfiles se equivoca más

## Parte 6 - Interpretación y conclusiones
### Ejercicio 16. Analizar resultados hipotéticos
| Modelo | Accuracy train | Accuracy validation |
|---|---:|---:|
| A | 0.99 | 0.61 |
| B | 0.83 | 0.81 |
| C | 0.55 | 0.53 |

Responde: ¿Cuál muestra indicios de sobreajuste? ¿Cuál es más consistente? ¿Cuál podría estar infraajustado? ¿Elegirías el de mayor accuracy de entrenamiento? ¿Qué comprobarías antes de decidir?

**Respuesta:** El **Modelo A** muestra claros indicios de sobreajuste (*overfitting*), ya que memoriza casi al 100% el entrenamiento (0.99) pero cae en validación (0.61). El **Modelo B** es el más consistente al mantener un buen rendimiento y muy cercano entre train (0.83) y validation (0.81). El **Modelo C** está infraajustado (*underfitting*), porque rinde mal en ambos conjuntos (0.55 y 0.53, peor incluso que predecir siempre la clase mayoritaria). No elegiría el de mayor accuracy en train porque no mide la capacidad real de generalizar; antes de decidir revisaría la matriz de confusión, precisión, *recall* y F1-score sobre validación